In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

from pathlib import Path

SEED = 42
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data' / 'mental_health'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device in use: {DEVICE}')

Device in use: cuda


In [2]:
df = pd.read_csv(DATA_DIR / 'mental_health.csv')
df

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,female,6.8,Instagram,6.6,2.0,2.76,1.0,low,3,4,4,0
1196,16,male,2.3,Both,8.0,1.9,2.12,0.4,high,7,4,4,0
1197,14,female,1.7,Both,8.7,0.7,3.98,0.8,high,1,1,1,0
1198,15,male,3.9,Both,8.5,2.1,3.19,0.6,high,7,9,9,0


In [3]:
# See if there are any missing values.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 122.0 KB


In [4]:
# Encoding categorical variables.
gender_map = {'male': 0, 'female': 1}
df['gender'] = df['gender'].map(gender_map)

interaction_map = {
    'low': 0,
    'medium': 1,
    'high': 2
}
df['social_interaction_level'] = df['social_interaction_level'].map(interaction_map)

df = pd.get_dummies(df, columns=['platform_usage'], prefix='platform', drop_first=True, dtype=int)
df

,age,gender,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label,platform_Instagram,platform_TikTok
0,14,0,7.9,7.4,2.9,3.01,1.5,0,2,2,1,0,1,0
1,19,1,1.9,8.0,2.9,3.22,0.8,2,8,1,10,0,0,1
2,17,1,1.3,7.6,0.5,3.92,0.0,2,2,4,2,0,1,0
3,15,0,7.4,6.9,1.6,3.48,0.8,1,1,7,9,0,0,1
4,15,1,4.7,4.9,3.0,2.37,1.4,1,3,5,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,1,6.8,6.6,2.0,2.76,1.0,0,3,4,4,0,1,0
1196,16,0,2.3,8.0,1.9,2.12,0.4,2,7,4,4,0,0,0
1197,14,1,1.7,8.7,0.7,3.98,0.8,2,1,1,1,0,0,0
1198,15,0,3.9,8.5,2.1,3.19,0.6,2,7,9,9,0,0,0


In [5]:
X = np.array(df.drop(columns='depression_label'))
y = np.array(df['depression_label'])
print(f'X: {X.shape}\ny: {y.shape}')

X: (1200, 13)
y: (1200,)


In [6]:
# Check if the output is only mede of integers and how many modalities there are.
np.unique(y, return_counts=True)

(array([0, 1]), array([1169,   31]))

In [7]:
# Standard scaling only X.
for var in range(X.shape[1]):
    mean = np.mean(X[:, var])
    std = np.std(X[:, var])
    X[:, var] = (X[:, var] - mean) / std

In [8]:
# Split into train and test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=SEED)

In [9]:
# Transform into tensors.
X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor(y_test)
X_train.size()

torch.Size([960, 13])

In [10]:
# Build dataset and set up data loader.
dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

In [11]:
# Define neural network structure.
class MentalHealthNet(nn.Module):
    def __init__(self):
        super(MentalHealthNet, self).__init__()
        
        # First layer: 12 --> 256 because dataset has 13 features.
        self.layer1 = nn.Linear(13, 256)
        
        # 2 additional layers in a pyramid shape.
        self.layer2 = nn.Linear(256, 196)
        self.layer3 = nn.Linear(196, 64)
        
        # Output layer with size 2 (output is either 1 or 0 in this dataset).
        self.output_layer = nn.Linear(64, 2)
        
        # Activation function is ReLU in this case.
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x

In [12]:
# Send the model to the right device.
model = MentalHealthNet().to(DEVICE)

In [13]:
# Decide which loss and optimizer to use.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

In [14]:
# Construct training loop.
n_epochs = 200
for epoch in range(n_epochs):
    # ── Training phase ───────────────────────────────────────────────────────
    model.train()
    # Initialization.
    total_loss = 0
    total_samples = 0
    total_correct = 0
    
    # Iterate through the batches based on batch size.
    for inputs, labels in loader:
        # Send the material to the correct device.
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Zero out the gradients of the previous batch.
        optimizer.zero_grad()
        
        # Make predictions.
        outputs = model(inputs)
        
        # Calculate loss with the chosen function.
        batch_loss = criterion(outputs, labels)
        
        # Chain rule.
        batch_loss.backward()
        
        # Optimize neurons.
        optimizer.step()
        
        # Evaluate.
        total_loss += batch_loss
        _, pred = torch.max(outputs, 1)
        total_samples += labels.size(0)
        total_correct += (pred == labels).sum().item()
        
    # ── Test phase ─────────────────────────────────────────────────────
    model.eval()    # Set the model to evaluation mode.
    
    # Initialization.
    test_loss = 0
    test_correct = 0
    test_samples = 0
    
    # Iterate through the batches again.
    with torch.no_grad():    # Saves time.
        for test_inputs, test_labels in test_loader:
            # Send to device.
            test_inputs, test_labels = test_inputs.to(DEVICE), test_labels.to(DEVICE)
            
            # Skip directly do outputs calculation.
            test_outputs = model(test_inputs)
            
            # Calculate loss.
            test_batch_loss = criterion(test_outputs, test_labels)
            
            # Evaluate.
            test_loss += test_batch_loss
            _, test_pred = torch.max(test_outputs, 1)
            test_samples += test_labels.size(0)
            test_correct += (test_pred == test_labels).sum().item()
        
    # ── Epoch summary ────────────────────────────────────────────────────────
    train_acc = total_correct / total_samples
    test_acc = test_correct / test_samples
    print(f'Epoch: [{epoch + 1:>3}/{n_epochs}] | '
          f'Training Loss: {total_loss>7:.4f}, Training Accuracy: {train_acc:.4f} | '
          f'Test Loss: {test_loss>7:.4f}, Test Accuracy: {test_acc:.4f}')

Epoch: [  1/200] | Training Loss: 1.0000, Training Accuracy: 0.8635 | Test Loss: 2.3463, Test Accuracy: 0.9750
Epoch: [  2/200] | Training Loss: 1.0000, Training Accuracy: 0.9740 | Test Loss: 1.8839, Test Accuracy: 0.9750
Epoch: [  3/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 1.3515, Test Accuracy: 0.9750
Epoch: [  4/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.8768, Test Accuracy: 0.9750
Epoch: [  5/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.6091, Test Accuracy: 0.9750
Epoch: [  6/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.5073, Test Accuracy: 0.9750
Epoch: [  7/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.4724, Test Accuracy: 0.9750
Epoch: [  8/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.4537, Test Accuracy: 0.9750
Epoch: [  9/200] | Training Loss: 0.0000, Training Accuracy: 0.9740 | Test Loss: 0.4391, Test Accuracy: 0.9750
E